# Hyperparameter Tuning in Random Forest

This notebook covers:
- What hyperparameters are
- Why tuning is required
- GridSearchCV
- RandomizedSearchCV
- Comparison between default vs tuned model

## 1. Load Dataset (Iris from Kaggle CSV)

In [ ]:

import pandas as pd

df = pd.read_csv("Iris.csv")
df.drop(columns=['Id'], inplace=True)
df.head()

## 2. Encode Target Variable

In [ ]:

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Species'] = le.fit_transform(df['Species'])

X = df.drop('Species', axis=1)
y = df['Species']
df

## 3. Train-Test Split

In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 4. Baseline Random Forest (No Tuning)

In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_default = RandomForestClassifier(random_state=42) #default values ....
rf_default.fit(X_train, y_train)

y_pred_default = rf_default.predict(X_test)
accuracy_score(y_test, y_pred_default)

## 5. Important Hyperparameters Explained

- n_estimators: number of trees
- max_depth: depth of each tree
- min_samples_split: minimum samples to split a node (dont split a branch where no of samples/datapoints are less than 10)
- min_samples_leaf: minimum samples at leaf node

## 6. GridSearchCV (Exhaustive Search)

try every possible combination of hyperparameters

Selects best performing set

CV: 5-fold cross-validation

Each combination is tested on:

4 folds (train)

1 fold (validation)

In [ ]:

from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [5, 10, 30, 50, 100],
    'max_depth': [5, 10],
    'min_samples_split': [10, 20, 30],  #min sample needed to split a node
    'min_samples_leaf': [8, 12, 15] #min sample at leaf node
}

rf = RandomForestClassifier(random_state=42)

#GridSearchCV: try every possible combination 3*3*3*3=81 models
#uses cross validation
#n_jobs=-1: use all cores
#Select best performing set

grid_search = GridSearchCV(
    estimator=rf, #model
    param_grid=param_grid, #grid
    cv=5, #5 fold cross validation
    scoring='accuracy', #average accuracy
    n_jobs=-1
)

# hyperparamete sets.....cv
grid_search.fit(X_train, y_train) #train mutiple models, compare accuracy and save the best one

### Best Parameters from GridSearch

In [ ]:
grid_search.best_params_ #attribute #besr average accuracy

### Evaluate Tuned Model

In [ ]:

best_rf_grid = grid_search.best_estimator_ #load the best trained model out of all the models trained

y_pred_grid = best_rf_grid.predict(X_test)
accuracy_score(y_test, y_pred_grid) #not average number

## 7. RandomizedSearchCV 

Random Search is a **hyperparameter tuning technique** used to find good model settings **without trying every possible combination**.

Instead of checking *all* combinations (like Grid Search), it:
👉 **randomly picks combinations**
👉 trains the model
👉 evaluates performance
👉 keeps the best one

---

## 1️⃣ Why Hyperparameter Tuning?

Models like **Random Forest** have parameters such as:
- number of trees
- depth of trees
- minimum samples per split

These are **not learned from data** — we must choose them.

Bad values → poor accuracy  
Good values → better generalization  

---



In [ ]:

from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_dist = {
    'n_estimators': np.arange(50, 300, 50), #50, 100, 150, 200, 250  → 5 values
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
rf = RandomForestClassifier(random_state=42)
random_search = RandomizedSearchCV(
    estimator=rf, #The model we are tuning
    param_distributions=param_dist, #The range of values Random Search can choose from
    n_iter=10, #Random Search will try only 10 random combinations not all 180(5 × 4 × 3 × 3 = 180)(param_dist)
    cv=5, # 5-fold cross-validation
    scoring='accuracy', #Metric used to compare models
    random_state=42, #Ensures same random combinations every run
    #n_jobs=-1 #Uses all CPU cores #Faster execution
)

random_search.fit(X_train, y_train)


### Best Parameters from RandomSearch

In [ ]:

random_search.best_params_


### Evaluate Random Search Model

In [ ]:

best_rf_random = random_search.best_estimator_

y_pred_random = best_rf_random.predict(X_test)
accuracy_score(y_test, y_pred_random)


## 8. GridSearch vs RandomSearch Summary

GridSearch:
- Tries all combinations
- Slow but thorough

RandomSearch:
- Tries random combinations
- Faster and scalable


Recommendation:
👉 Use RandomSearch first, then refine with GridSearch

# Grid Search vs Random Search: Which Is Better?

❌ Grid Search is NOT always better  
✅ The better choice depends on the problem

---

## When Grid Search Is Better ✅

Grid Search is better when:

- Dataset is small
- Very few hyperparameters
- Few values per hyperparameter
- Model training is fast
- You want a guaranteed best result **within a small search space**

📌 Reason: Grid Search tries **every possible combination**

---

## When Random Search Is Better 🚀

Random Search is better when:

- Dataset is large
- Many hyperparameters
- Wide range of values
- Model training is expensive
- Time or compute resources are limited

📌 Reason: Random Search explores **more diverse combinations** in less time

---

## Important Truth ⭐ (Exam & Interview)

> Grid Search finds the best model **only inside the grid you define**.  
> If the grid is poorly chosen, the best result is still suboptimal.

---

## Practical Comparison

| Situation | Better Choice |
|---------|--------------|
| Few hyperparameters | Grid Search |
| Many hyperparameters | Random Search |
| Limited time | Random Search |
| Teaching / demo | Grid Search |
| Real-world ML problems | Random Search |
| Guaranteed optimum in small space | Grid Search |

---

## Simple Analogy 🧠

### Grid Search
> Search every house in one small lane

### Random Search
> Randomly search houses across the entire city

---

## Final One-Line Answer 🎯

> **Grid Search guarantees the best result only within a predefined grid, while Random Search is more efficient and practical for large hyperparameter spaces.**
